# Silver — Orders Transform + Reconciliation

Transform `bronze.orders` → `silver.orders`, route late arrivals, quality gate, and reconcile bronze vs silver.

In [ ]:
import sys
import importlib

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = notebook_path.split("/notebooks/")[0]
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import src.transformations.silver_orders as silver_orders_module
importlib.reload(silver_orders_module)

from pyspark.sql import functions as F
from src.transformations.silver_orders import SilverOrdersConfig, build_silver_orders, split_late_arrivals
from src.quality.engine import get_failed_records
from src.quality.dlq import route_to_dlq
from src.quality.rules import ORDERS_DQ_RULES
from src.quality.reconciliation import run_reconciliation, RECONCILIATION_LOG_TABLE
from src.ingestion.idempotent_loader import save_run_report_to_volume

config = SilverOrdersConfig()
print("Loaded from:", silver_orders_module.__file__)

In [ ]:
# Deduped bronze slice for silver quality gate
from pyspark.sql.window import Window

bronze = spark.table(config.bronze_table)
w = Window.partitionBy("order_id").orderBy(
    F.col("order_channel").isNull(),
    F.col("_ingested_at").desc_nulls_last(),
)
bronze_deduped = (
    bronze.withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

failed = get_failed_records(spark, bronze_deduped, ORDERS_DQ_RULES, severities=("critical",))
route_to_dlq(spark, failed, source_table=config.bronze_table, failure_reason="silver_orders_gate")
valid_keys = bronze_deduped.join(failed.select("order_id"), on="order_id", how="left_anti")
print("Valid orders for silver:", valid_keys.count())

In [ ]:
# Build and write silver orders
silver = build_silver_orders(spark, config, source_df=valid_keys)
on_time, late = split_late_arrivals(silver, config)

on_time.write.format("delta").mode("overwrite").saveAsTable(config.silver_table)
late.write.format("delta").mode("overwrite").saveAsTable(config.late_arrivals_table)
print("Silver orders:", spark.table(config.silver_table).count())
print("Late arrivals:", spark.table(config.late_arrivals_table).count())

In [ ]:
display(
    spark.table(config.silver_table)
    .groupBy("arrival_status")
    .count()
    .orderBy("arrival_status")
)

In [ ]:
# Reconciliation bronze (deduped) vs silver
source = bronze_deduped.select("order_id")
target = spark.table(config.silver_table).select("order_id")
recon_id = run_reconciliation(
    spark, source, target,
    source_table=config.bronze_table,
    target_table=config.silver_table,
    key_column="order_id",
)
display(spark.table(RECONCILIATION_LOG_TABLE).filter(F.col("reconciliation_id") == recon_id))

In [ ]:
import json

silver_count = spark.table(config.silver_table).count()
late_count = spark.table(config.late_arrivals_table).count()
arrival_on_time = {
    row["arrival_status"]: row["count"]
    for row in spark.table(config.silver_table).groupBy("arrival_status").count().collect()
}
arrival_late = {
    row["arrival_status"]: row["count"]
    for row in spark.table(config.late_arrivals_table).groupBy("arrival_status").count().collect()
}
report = {
    "silver_table": config.silver_table,
    "late_arrivals_table": config.late_arrivals_table,
    "silver_orders_count": silver_count,
    "late_arrivals_count": late_count,
    "reconciliation_id": recon_id,
    "arrival_status_on_time_table": arrival_on_time,
    "arrival_status_late_arrivals_table": arrival_late,
}
print(json.dumps(report, indent=2))
try:
    save_run_report_to_volume(report, dbutils, "/Volumes/globalmart/metadata/run_reports/silver_orders_latest.json")
except Exception as exc:
    print("Volume save skipped:", exc)